## 1. Download and parse the data

***!!!Set your TFL key using an invironment vairable!!!***

on windows: command is `set TFL_API_KEY=your_key_here` in command prompt or `$env:TFL_API_KEY = "your_key_here"` in powershell or set it in windows
get your key at (TFL API registration)[https://api-portal.tfl.gov.uk/]

In [8]:
!pip install requests pandas openpyxl tqdm networkx plotly nbformat

import os, json, time, math, requests
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

TFL_API_KEY = os.environ.get("TFL_API_KEY", "YOUR_KEY_HERE")

def download(url, dest, chunk=8192):
    """GET url → dest, skipping if already present."""
    if dest.exists():
        print(f"  [skip] {dest.name}")
        return dest
    print(f"  [GET]  {url}")
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True,
                                      desc=dest.name, leave=False) as bar:
        for data in r.iter_content(chunk):
            f.write(data)
            bar.update(len(data))
    print(f"  [OK]   {dest}")
    return dest

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\david\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


We then download the station data before getting their crowding data for each station (this is the step that requires the TfL key)

1. We use Oliver O'Brien's GeoJSON files (derived from OpenStreetMap) which provide every TfL station's coordinates and the ordered stop sequences for each line branch.
2. Annual Station Entry/Exit Counts - We download three years for comparison directly from TfL:
- 2019 — pre-COVID baseline (peak network usage)
- 2022 — COVID recovery year
- 2023 — most recent complete year

In [9]:
# Structure of TfL stations
BASE = "https://raw.githubusercontent.com/oobrien/vis/master/tubecreature/data"
download(f"{BASE}/tfl_stations.json", DATA_DIR / "tfl_stations.json")
download(f"{BASE}/tfl_lines.json",    DATA_DIR / "tfl_lines.json")

# Download crowding data
AC = "https://crowding.data.tfl.gov.uk/Annual%20Station%20Counts"
annual_urls = {
    "2019": (f"{AC}/2019/AnnualisedEntryExit_2019.xlsx",   "annual_counts_2019.xlsx"),
    "2022": (f"{AC}/2022/AC2022_AnnualisedEntryExit.xlsx", "annual_counts_2022.xlsx"),
    "2023": (f"{AC}/2023/AC2023_AnnualisedEntryExit.xlsx", "annual_counts_2023.xlsx"),
}
for yr, (url, fname) in annual_urls.items():
    download(url, DATA_DIR / fname)

crowding_path = DATA_DIR / "station_crowding.json"

# Check if existing file is valid (non-empty list)
if crowding_path.exists():
    with open(crowding_path) as f:
        _cr = json.load(f)
    if len(_cr) == 0:
        print("  [stale] station_crowding.json is empty — deleting and re-fetching...")
        crowding_path.unlink()

if crowding_path.exists():
    print(f"  [skip] {crowding_path.name}")
elif TFL_API_KEY == "YOUR_KEY_HERE":
    print("Warning: TFL_API_KEY not set — skipping crowding download.")
else:
    TFL_LINES = [
        "bakerloo", "central", "circle", "district", "hammersmith-city",
        "jubilee", "metropolitan", "northern", "piccadilly", "victoria",
        "waterloo-city"
    ]

    print("Fetching station list")
    stops = []
    for line in TFL_LINES:
        r = requests.get(f"https://api.tfl.gov.uk/Line/{line}/StopPoints",
                         params={"app_key": TFL_API_KEY}, timeout=30)
        r.raise_for_status()
        stops.extend(r.json())
        time.sleep(0.1)

    seen, stations = set(), []
    for s in stops:
        nid = s.get("naptanId", "")
        if nid and nid not in seen:
            seen.add(nid)
            stations.append({
                "naptanId": nid,
                "name":     s.get("commonName", ""),
                "lat":      s.get("lat"),
                "lon":      s.get("lon"),
            })
    print(f"  Found {len(stations)} unique stations")

    results = []
    for st in tqdm(stations, desc="Crowding API"):
        r = requests.get(
            f"https://api.tfl.gov.uk/crowding/{st['naptanId']}",
            params={"app_key": TFL_API_KEY}, timeout=15
        )
        crowding = r.json() if r.status_code == 200 else {}
        results.append({**st, "crowding": crowding})
        time.sleep(0.1)

    with open(crowding_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  [OK]   {len(results)} stations → {crowding_path}")

  [skip] tfl_stations.json
  [skip] tfl_lines.json
  [skip] annual_counts_2019.xlsx
  [skip] annual_counts_2022.xlsx
  [skip] annual_counts_2023.xlsx
  [skip] station_crowding.json


Now we parse the data, we need to flat the geodata into a df (one row per station)

Then we actually construct the graph:
We first collapse the pollylines into "station-to-station" edges.
Approach:
1. Build a lookup: for each station, find the nearest coordinate in the GeoJSON
2. Walk each branch's coordinate list, every time we hit a coordinate that
   snaps to a station, record an edge from the previous snapped station
3. Compute Haversine distance and estimated travel time for each edge
Average tube speed ~33 km/h gives `travel_time_min ≈ dist_km / 33 × 60`

In [10]:
# Flatten GeoJSON into df
with open(DATA_DIR / "tfl_stations.json") as f:
    stations_geo = json.load(f)

stations_df = pd.DataFrame([
    {
        "station_id": feat["properties"]["id"],
        "name":       feat["properties"]["name"],
        "zone":       feat["properties"].get("zone"),
        "lines":      [l["name"] for l in feat["properties"].get("lines", [])],
        "lon":        feat["geometry"]["coordinates"][0],
        "lat":        feat["geometry"]["coordinates"][1],
    }
    for feat in stations_geo["features"]
])

print(f"stations_df: {stations_df.shape}")
print(stations_df.head(3).to_string())

# Construct edges
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat, dlon = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = (math.sin(dlat/2)**2
         + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2))
         * math.sin(dlon/2)**2)
    return R * 2 * math.asin(math.sqrt(a))

AVG_TUBE_SPEED_KMH = 33.0
SNAP_THRESHOLD_KM  = 0.15

# Build (lat, lon, station_id) lookup
station_coords = stations_df[["station_id", "name", "lat", "lon"]].values

def snap_to_station(lat, lon):
    """Return (station_id, name) of nearest station within threshold, else None."""
    best_id, best_name, best_d = None, None, float("inf")
    for sid, sname, slat, slon in station_coords:
        d = haversine_km(lat, lon, slat, slon)
        if d < best_d:
            best_d, best_id, best_name = d, sid, sname
    if best_d <= SNAP_THRESHOLD_KM:
        return best_id, best_name
    return None, None

with open(DATA_DIR / "tfl_lines.json") as f:
    lines_geo = json.load(f)

edges = []
for feat in lines_geo["features"]:
    props     = feat["properties"]
    line_name = props["lines"][0]["name"] if props.get("lines") else "Unknown"
    coords    = feat["geometry"]["coordinates"]

    prev_sid, prev_name = None, None
    prev_lat, prev_lon  = None, None

    for lon, lat in coords:
        sid, sname = snap_to_station(lat, lon)
        if sid is not None:
            if prev_sid is not None and sid != prev_sid:
                dist = haversine_km(prev_lat, prev_lon, lat, lon)
                edges.append({
                    "line":            line_name,
                    "station_a_id":    prev_sid,
                    "station_a":       prev_name,
                    "station_b_id":    sid,
                    "station_b":       sname,
                    "dist_km":         round(dist, 4),
                    "travel_time_min": round(dist / AVG_TUBE_SPEED_KMH * 60, 2),
                })
            prev_sid, prev_name = sid, sname
            prev_lat, prev_lon  = lat, lon

edges_df = pd.DataFrame(edges).drop_duplicates(
    subset=["line", "station_a_id", "station_b_id"]
)
# Drop self-loops (same station matched under two different IDs)
edges_df = edges_df[edges_df["station_a"] != edges_df["station_b"]].copy()
# Drop implausibly short edges (< 0.2 min)
edges_df = edges_df[edges_df["travel_time_min"] >= 0.2].copy()

print(f"edges_df: {edges_df.shape}")
print(f"Lines:    {sorted(edges_df['line'].unique())}")
print(f"\nTravel time stats (mins):\n{edges_df['travel_time_min'].describe().round(2)}")
print(f"\nFirst 5 edges:")
print(edges_df.head(5).to_string())

stations_df: (552, 6)
    station_id          name zone                           lines       lon        lat
0  940GZZLUACT    Acton Town    3          [District, Piccadilly] -0.279917  51.502644
1  940GZZLUACY       Archway  2/3                      [Northern] -0.134745  51.565371
2  940GZZLUADE  Aldgate East    1  [District, Hammersmith & City] -0.071877  51.515147
edges_df: (553, 7)
Lines:    ['Bakerloo', 'Central', 'Circle', 'Crossrail 2', 'DLR', 'District', 'Elizabeth line', 'IFS Cloud Cable Car', 'Jubilee', 'London Overground', 'Metropolitan', 'Northern', 'Piccadilly', 'Thameslink 6tph line', 'Tramlink', 'Victoria', 'Waterloo & City']

Travel time stats (mins):
count    553.00
mean       1.97
std        1.68
min        0.20
25%        0.89
50%        1.59
75%        2.54
max       19.53
Name: travel_time_min, dtype: float64

First 5 edges:
                line station_a_id        station_a station_b_id        station_b  dist_km  travel_time_min
1  London Overground  940GZZLUBLG  

Parse our Annual counts into a df and clean up data
Then we standardise the station code column before aligning the days columns as the data from 2019 is in a different format

In [11]:
DAY_COLS_4 = ["mon", "twt", "fri", "sat"]           # 2019
DAY_COLS_5 = ["mon", "twt", "fri", "sat", "sun"]    # 2022/23

# TfL files use duplicate column headers so we need to renamne them
def rename_annual_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Rename auto-generated entries.N / exits.N columns to meaningful names."""
    cols = list(df.columns)
    entry_idx = [i for i, c in enumerate(cols) if c.startswith("entries")]
    exit_idx  = [i for i, c in enumerate(cols) if c.startswith("exits")]
    enex_idx  = [i for i, c in enumerate(cols) if c.startswith("en_ex")]

    day_cols = DAY_COLS_4 if len(entry_idx) == 4 else DAY_COLS_5

    rename = {}
    for i, idx in enumerate(entry_idx):
        rename[cols[idx]] = f"entries_{day_cols[i]}"
    for i, idx in enumerate(exit_idx):
        rename[cols[idx]] = f"exits_{day_cols[i]}"

    # 2019 has 1 aggregate; 2022/23 have 3
    agg_names = ["en_ex_weekly", "en_ex_12week", "en_ex_annual"]
    if len(enex_idx) == 1:
        rename[cols[enex_idx[0]]] = "en_ex_annual"
    else:
        for i, idx in enumerate(enex_idx):
            rename[cols[idx]] = agg_names[i] if i < len(agg_names) else f"en_ex_{i}"

    df = df.rename(columns=rename)

    # Standardise station-code column name (nlc → mnlc for 2019)
    if "nlc" in df.columns and "mnlc" not in df.columns:
        df = df.rename(columns={"nlc": "mnlc", "asc": "masc"})

    # Cast all numeric columns (2023 loads as object/string)
    num_cols = [c for c in df.columns if c not in
                ["mode", "masc", "station", "coverage", "source"]]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


def parse_annual_counts(path: Path) -> pd.DataFrame:
    xl = pd.ExcelFile(path)
    data_sheet = None
    for s in xl.sheet_names:
        probe = pd.read_excel(path, sheet_name=s, header=None, nrows=10)
        if probe[0].astype(str).str.strip().eq("Mode").any():
            data_sheet = s
            break
    if data_sheet is None:
        raise ValueError(f"No data sheet in {path.name}. Sheets: {xl.sheet_names}")

    raw = pd.read_excel(path, sheet_name=data_sheet, header=None)
    header_row = int(raw[raw[0].astype(str).str.strip() == "Mode"].index[0])

    df = pd.read_excel(path, sheet_name=data_sheet, header=header_row)
    df = df.dropna(how="all").dropna(axis=1, how="all")
    first_col = df.columns[0]
    valid_modes = {"LU", "LO", "DLR", "TfL Rail", "EL"}
    df = df[df[first_col].astype(str).str.strip().isin(valid_modes)].copy()
    df.columns = [str(c).strip().lower().replace(" ", "_").replace("/", "_")
                  for c in df.columns]
    df = rename_annual_cols(df)
    return df

annual_dfs = {}
for yr, (_, fname) in annual_urls.items():
    df = parse_annual_counts(DATA_DIR / fname)
    annual_dfs[yr] = df
    print(f"\nannual_dfs['{yr}']: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    # Show station + first entry col + last col (annualised total), whatever they're called
    entry_cols = [c for c in df.columns if c.startswith("entries_")]
    last_col   = df.columns[-1]
    show_cols  = ["station"] + entry_cols[:2] + [last_col]
    print(df[show_cols].head(3).to_string())


annual_dfs['2019']: (425, 15)
Columns: ['mode', 'mnlc', 'masc', 'station', 'coverage', 'source', 'entries_mon', 'entries_twt', 'entries_fri', 'entries_sat', 'exits_mon', 'exits_twt', 'exits_fri', 'exits_sat', 'en_ex_annual']
        station  entries_mon  entries_twt  en_ex_annual
0    Acton Town        10091        10240  6.186555e+06
1       Aldgate        17039        15532  9.956600e+06
2  Aldgate East        23063        24584  1.414865e+07

annual_dfs['2022']: (427, 19)
Columns: ['mode', 'mnlc', 'masc', 'station', 'coverage', 'source', 'entries_mon', 'entries_twt', 'entries_fri', 'entries_sat', 'entries_sun', 'exits_mon', 'exits_twt', 'exits_fri', 'exits_sat', 'exits_sun', 'en_ex_weekly', 'en_ex_12week', 'en_ex_annual']
        station   entries_mon   entries_twt  en_ex_annual
0    Acton Town   7304.888889   8194.000000  4.931972e+06
1       Aldgate  10032.666667  12378.444444  6.902494e+06
2  Aldgate East  15100.500000  17401.777778  1.022949e+07

annual_dfs['2023']: (427, 19)
C

Finally, we can parse the crowding JSON data into a df

In [12]:

crowding_df = pd.DataFrame()

if crowding_path.exists():
    with open(crowding_path) as f:
        cr = json.load(f)

    # Print a raw sample so we can verify the structure
    # sample = next((s for s in cr if s.get("crowding")), None)
    # if sample:
    #     print("Sample crowding entry:")
    #     print(json.dumps(sample, indent=2)[:600])

    rows = []
    for s in cr:
        base = {
            "naptanId": s["naptanId"],
            "name":     s["name"],
            "lat":      s["lat"],
            "lon":      s["lon"],
        }
        crowding = s.get("crowding", {})

        # API returns a dict with a 'timetable' or 'dayOfWeek' list
        # Normalise both possible shapes into a flat list of day entries
        day_entries = crowding.get("daysOfWeek", []) if isinstance(crowding, dict) else []

        if day_entries:
            for day_entry in day_entries:
                day = day_entry.get("dayOfWeek", "")
                for ts in day_entry.get("timeBands", []):
                    rows.append({
                        **base,
                        "day":       day,
                        "time_band": ts.get("timeBand"),
                        "busyness":  ts.get("percentageOfBaseLine"),
                    })
        else:
            # Station exists but no crowding data available
            rows.append({**base, "day": None, "time_band": None, "busyness": None})

    crowding_df = pd.DataFrame(rows)
    n_with = crowding_df["busyness"].notna().sum()
    print(f"\ncrowding_df: {crowding_df.shape} — {n_with} rows with busyness data")
    print(crowding_df[crowding_df["busyness"].notna()].head(5).to_string())
else:
    print("station_crowding.json not found.")


crowding_df: (182114, 7) — 182112 rows with busyness data
      naptanId                              name        lat      lon  day    time_band  busyness
0  940GZZLUBST  Baker Street Underground Station  51.522883 -0.15713  TUE  00:00-00:15      0.02
1  940GZZLUBST  Baker Street Underground Station  51.522883 -0.15713  TUE  00:15-00:30      0.02
2  940GZZLUBST  Baker Street Underground Station  51.522883 -0.15713  TUE  00:30-00:45      0.02
3  940GZZLUBST  Baker Street Underground Station  51.522883 -0.15713  TUE  00:45-01:00      0.01
4  940GZZLUBST  Baker Street Underground Station  51.522883 -0.15713  TUE  01:00-01:15      0.01


## 2. Compute Node Weights and Build Graph

Compute Node Weights
Merge 2023 annual ridership onto stations_df. Where a station appears under multiple modes (e.g. Bank in both LU and DLR) we sum across all modes.

In [13]:
ridership = (
    annual_dfs["2023"]
    .groupby("station")["en_ex_annual"]
    .sum()
    .reset_index()
    .rename(columns={"station": "name", "en_ex_annual": "ridership_2023"})
)

# Normalise names for joining
ridership["name_key"]   = ridership["name"].str.strip().str.lower()
stations_df["name_key"] = stations_df["name"].str.strip().str.lower()

stations_df = stations_df.merge(
    ridership[["name_key", "ridership_2023"]], on="name_key", how="left"
)

# Fill missing ridership with 0
stations_df["ridership_2023"] = stations_df["ridership_2023"].fillna(0)

matched = (stations_df["ridership_2023"] > 0).sum()
print(f"Stations matched to ridership data: {matched} / {len(stations_df)}")
print(stations_df[["name", "ridership_2023"]].nlargest(5, "ridership_2023").to_string())

Stations matched to ridership data: 364 / 552
                         name  ridership_2023
129  King's Cross St. Pancras      72124262.0
229      Tottenham Court Road      58726155.0
503      Tottenham Court Road      58726155.0
217                 Stratford      54376225.0
485                 Stratford      54376225.0


Now we actually build the graph
Each node stores coordinates and ridership; each edge stores line and travel time.

In [14]:
import networkx as nx

G = nx.MultiGraph()

for _, row in stations_df.iterrows():
    G.add_node(
        row["station_id"],
        name=row["name"],
        lat=row["lat"],
        lon=row["lon"],
        zone=str(row.get("zone", "?")),
        ridership=float(row["ridership_2023"]),
    )

for _, row in edges_df.iterrows():
    if row["station_a_id"] in G and row["station_b_id"] in G:
        G.add_edge(
            row["station_a_id"],
            row["station_b_id"],
            line=row["line"],
            travel_time_min=row["travel_time_min"],
            dist_km=row["dist_km"],
        )

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Connected components: {nx.number_connected_components(G)}")

# Top 5 nodes by ridership
top5 = sorted(G.nodes(data=True), key=lambda x: x[1]["ridership"], reverse=True)[:5]
print("\nTop 5 stations by ridership:")
for sid, d in top5:
    print(f"  {d['name']:35s}  {d['ridership']:>12,.0f}")

Graph built: 552 nodes, 553 edges
Connected components: 68

Top 5 stations by ridership:
  King's Cross St. Pancras               72,124,262
  Tottenham Court Road                   58,726,155
  Tottenham Court Road                   58,726,155
  Stratford                              54,376,225
  Stratford                              54,376,225


We will make a interactive map visualisation to make sure everything has worked as expected

Edges are coloured by line, thickness proportional to 1/travel_time so shorter (busier) links appear thicker
Nodes are sized and coloured by 2023 ridership

In [ ]:
import plotly.graph_objects as go

LINE_COLOURS = {
    "Bakerloo":             "#B36305",
    "Central":              "#E32017",
    "Circle":               "#FFD300",
    "District":             "#00782A",
    "Hammersmith & City":   "#F3A9BB",
    "Jubilee":              "#A0A5A9",
    "Metropolitan":         "#9B0056",
    "Northern":             "#000000",
    "Piccadilly":           "#003688",
    "Victoria":             "#0098D4",
    "Waterloo & City":      "#95CDBA",
    "DLR":                  "#00A4A7",
    "Elizabeth line":       "#6950A1",
    "London Overground":    "#EE7C0E",
    "Tramlink":             "#84B817",
    "Crossrail 2":          "#AB008B",
    "Thameslink 6tph line": "#D42E82",
    "IFS Cloud Cable Car":  "#E21836",
    "East London":          "#EE7C0E",
}

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Connected components: {nx.number_connected_components(G)}")

# Top 5 nodes by ridership
top5 = sorted(G.nodes(data=True), key=lambda x: x[1]["ridership"], reverse=True)[:5]
print("\nTop 5 stations by ridership:")
for sid, d in top5:
    print(f"  {d['name']:35s}  {d['ridership']:>12,.0f}")

fig = go.Figure()

# ── Edges — one Scattermap trace per line ────────────────────────────────────
for line_name, colour in LINE_COLOURS.items():
    line_edges = edges_df[edges_df["line"] == line_name]
    if line_edges.empty:
        continue

    lats, lons, texts = [], [], []
    for _, e in line_edges.iterrows():
        a = G.nodes.get(e["station_a_id"])
        b = G.nodes.get(e["station_b_id"])
        if not a or not b:
            continue
        lats += [a["lat"], b["lat"], None]
        lons += [a["lon"], b["lon"], None]
        hover = (f"{e['station_a']} → {e['station_b']}<br>"
                 f"Line: {line_name}<br>"
                 f"Distance: {e['dist_km']:.2f} km<br>"
                 f"Travel time: {e['travel_time_min']:.1f} min")
        texts += [hover, hover, None]

    fig.add_trace(go.Scattermap(
        lat=lats, lon=lons,
        mode="lines",
        line=dict(color=colour, width=2),
        hovertext=texts, hoverinfo="text",
        name=line_name,
    ))

# ── Nodes ─────────────────────────────────────────────────────────────────────
node_data  = list(G.nodes(data=True))
node_lats  = [d["lat"]       for _, d in node_data]
node_lons  = [d["lon"]       for _, d in node_data]
node_rid   = [d["ridership"] for _, d in node_data]
node_hover = [
    f"<b>{d['name']}</b><br>Zone: {d['zone']}<br>"
    f"2023 ridership: {d['ridership']:,.0f}<br>"
    f"Connections: {G.degree(sid)}"
    for sid, d in node_data
]

max_rid   = max(r for r in node_rid if r > 0)
node_size = [5 + 20 * (r / max_rid) ** 0.4 for r in node_rid]

fig.add_trace(go.Scattermap(
    lat=node_lats, lon=node_lons,
    mode="markers",
    marker=dict(
        size=node_size,
        color=node_rid,
        colorscale="YlOrRd",
        colorbar=dict(title="2023 Annual<br>Ridership", thickness=15),
        cmin=0, cmax=max_rid,
        opacity=0.9,
    ),
    hovertext=node_hover,
    hoverinfo="text",
    name="Stations",
))

fig.update_layout(
    title="London Underground — Node & Edge Weights",
    map=dict(
        style="carto-positron",
        center=dict(lat=51.509, lon=-0.118),
        zoom=10,
    ),
    legend=dict(x=0, y=1, bgcolor="rgba(255,255,255,0.8)", font=dict(size=10)),
    height=800,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

Graph built: 552 nodes, 553 edges
Connected components: 68

Top 5 stations by ridership:
  King's Cross St. Pancras               72,124,262
  Tottenham Court Road                   58,726,155
  Tottenham Court Road                   58,726,155
  Stratford                              54,376,225
  Stratford                              54,376,225


Node weight  = 2023 annual ridership (node size + colour)
Edge weight  = travel time in minutes (stored in graph, edges coloured by line)
